<a href="https://colab.research.google.com/github/muzamil-saleem/XAI_LLM_IDS/blob/main/00_setup_kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [ ]:
!pip install -q transformers==4.38.0 datasets shap==0.45.0 bertviz
!pip install -q scikit-learn imbalanced-learn scipy statsmodels kaggle
print('All libraries installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 re

In [ ]:
from google.colab import files, drive
import os, shutil

# Step 1: Upload kaggle.json — click Choose Files when prompted
files.upload()

# Step 2: Move to correct location
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle API ready.')

# Step 3: Mount Google Drive
drive.mount('/content/drive')

# Step 4: Set your working folder
BASE = '/content/drive/MyDrive/XAI_LLM_IDS/'
print(f'BASE folder: {BASE}')

# Step 5: Create all subfolders inside XAI_LLM_IDS
for folder in ['data', 'models', 'predictions', 'shap',
               'plots', 'plots/shap', 'plots/attention',
               'results', 'paper_tables']:
    os.makedirs(BASE + folder, exist_ok=True)

print('All subfolders created.')

Saving kaggle.json to kaggle.json
Kaggle API ready.
Mounted at /content/drive
BASE folder: /content/drive/MyDrive/XAI_LLM_IDS/
All subfolders created.


In [ ]:
import os

# Create folders first
os.makedirs('/content/unsw_raw/', exist_ok=True)
os.makedirs('/content/cic_raw/', exist_ok=True)

# UNSW-NB15 — about 2 minutes
print('Downloading UNSW-NB15...')
!kaggle datasets download -d mrwellsdavid/unsw-nb15 -p /content/unsw_raw/ --unzip -q
print('UNSW-NB15 done.')
print(os.listdir('/content/unsw_raw/'))

# CIC-IDS2017 — about 5–8 minutes
print('\nDownloading CIC-IDS2017...')
!kaggle datasets download -d cicdataset/cicids2017 -p /content/cic_raw/ --unzip -q
print('CIC-IDS2017 done.')
print(os.listdir('/content/cic_raw/'))

Dataset URL: https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
License(s): unknown
UNSW-NB15 done.
['UNSW-NB15_4.csv', 'UNSW-NB15_1.csv', 'UNSW-NB15_LIST_EVENTS.csv', 'UNSW_NB15_testing-set.csv', 'NUSW-NB15_features.csv', 'UNSW_NB15_training-set.csv', 'UNSW-NB15_3.csv', 'UNSW-NB15_2.csv']

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
CIC-IDS2017 done.
[]


In [ ]:
import os

# Check UNSW-NB15
print('=== UNSW-NB15 files ===')
unsw_files = os.listdir('/content/unsw_raw/')
for f in unsw_files:
    size = os.path.getsize(f'/content/unsw_raw/{f}') / 1e6
    print(f'  {f}  —  {size:.1f} MB')

# Check CIC-IDS2017
print('\n=== CIC-IDS2017 files ===')
cic_files = os.listdir('/content/cic_raw/')
for f in cic_files:
    size = os.path.getsize(f'/content/cic_raw/{f}') / 1e6
    print(f'  {f}  —  {size:.1f} MB')

=== UNSW-NB15 files ===

=== CIC-IDS2017 files ===


In [ ]:
!kaggle datasets list --search "unsw-nb15"

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 10, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 763, in authenticate
    if self._authenticate_with_legacy_apikey():
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 808, in _authenticate_with_legacy_apikey
    raise ValueError("Error: Missing %s in configuration." % item)
ValueError: Error: Missing username in configuration.


In [ ]:
import os

# Create the kaggle folder
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

# Paste YOUR token in the quotes below (the KGAT_... value)
token = "KGAT_a6271d4280fbdcc34f463b549796f687"

# Save it to the access_token file
token_path = os.path.expanduser('~/.kaggle/access_token')
with open(token_path, 'w') as f:
    f.write(token)
os.chmod(token_path, 0o600)

print('Token saved.')

Token saved.


In [ ]:
!kaggle datasets list --search "unsw-nb15"

ref                                                            title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dhoogla/unswnb15                                               UNSW-NB15                                           12286680  2024-09-09 09:28:06.997000          16404         96                1  
dhoogla/nfunswnb15v2                                           NF-UNSW-NB15-V2                                     49538453  2023-01-13 20:36:09.737000          62890         11                1  
sankurisrinath/nf-unsw-nb15-v2csv                              NF-UNSW-NB15-V2.CSV                                 37060516  2023-06-22 05:28:02.280000           1314         30                1  
mrwellsdavid/un

Step 6 — Preprocess UNSW-NB15

In [ ]:
import pandas as pd, numpy as np, joblib, json
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

RAW = '/content/unsw_raw/'

# Load official train and test splits
train_df = pd.read_csv(RAW + 'UNSW_NB15_training-set.csv', low_memory=False)
test_df  = pd.read_csv(RAW + 'UNSW_NB15_testing-set.csv',  low_memory=False)
print(f'Train: {len(train_df):,} rows    Test: {len(test_df):,} rows')
print('Columns:', train_df.columns.tolist())

Train: 82,332 rows    Test: 175,341 rows
Columns: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


Step 7 — Define features and clean data

In [ ]:
# Define which columns are which
DROP  = ['id', 'attack_cat', 'label']
CAT   = ['proto', 'state', 'service']
NUM   = [c for c in train_df.columns if c not in DROP + CAT]
FEATS = CAT + NUM

print(f'Numeric features:     {len(NUM)}')
print(f'Categorical features: {len(CAT)}')
print(f'Total features used:  {len(FEATS)}')

# Step 1: Fill missing values
for df in [train_df, test_df]:
    df[NUM] = df[NUM].fillna(df[NUM].median())
    df[CAT] = df[CAT].fillna(df[CAT].mode().iloc[0])

# Step 2: Log transform skewed numeric columns
skewed = [c for c in NUM if train_df[c].skew() > 5.0]
print(f'\nSkewed features found: {len(skewed)}')
for df in [train_df, test_df]:
    df[skewed] = np.log1p(df[skewed])

# Step 3: MinMax scale — fit on train only
scaler = MinMaxScaler()
train_df[NUM] = scaler.fit_transform(train_df[NUM])
test_df[NUM]  = scaler.transform(test_df[NUM])
print('Scaling done.')

# Step 4: Encode labels
le = LabelEncoder()
train_df['label_enc'] = le.fit_transform(train_df['attack_cat'].str.strip())
test_df['label_enc']  = le.transform(test_df['attack_cat'].str.strip())
print('Classes found:', le.classes_)

# Step 5: Class weights
cw = compute_class_weight('balanced',
     classes=np.unique(train_df['label_enc']),
     y=train_df['label_enc'])
class_weights = dict(enumerate(cw))
print('\nClass weights:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls}: {class_weights[i]:.2f}')

Numeric features:     39
Categorical features: 3
Total features used:  42

Skewed features found: 21
Scaling done.
Classes found: ['Analysis' 'Backdoor' 'DoS' 'Exploits' 'Fuzzers' 'Generic' 'Normal'
 'Reconnaissance' 'Shellcode' 'Worms']

Class weights:
  Analysis: 12.16
  Backdoor: 14.12
  DoS: 2.01
  Exploits: 0.74
  Fuzzers: 1.36
  Generic: 0.44
  Normal: 0.22
  Reconnaissance: 2.36
  Shellcode: 21.78
  Worms: 187.12


Step 8 — Serialize records for DistilBERT

In [ ]:
# Convert each tabular row into a text string
# Example output: "proto tcp state FIN dur 0.121 spkts 6 dpkts 4 ..."

def serialize(row):
    parts = []
    for col in FEATS:
        val = row[col]
        if isinstance(val, float):
            val = f'{val:.3g}'
        parts.append(f'{col} {val}')
    return ' '.join(parts)

print('Serializing training set (takes ~1 min)...')
train_df['text'] = train_df.apply(serialize, axis=1)

print('Serializing test set...')
test_df['text'] = test_df.apply(serialize, axis=1)

print('Done. Example serialized record:')
print(train_df['text'].iloc[0][:200], '...')

Serializing training set (takes ~1 min)...
Serializing test set...
Done. Example serialized record:
proto udp state INT service - dur 2.68e-06 spkts 0.0473 dpkts 0 sbytes 0.225 dbytes 0 rate 0.0909 sttl 0.996 dttl 0 sload 0.849 dload 0 sloss 0 dloss 0 sinpkt 0.000994 dinpkt 0 sjit 0 djit 0 swin 0 st ...


Step 9 — Save everything to Drive

In [ ]:
print('Saving to Drive...')

train_df.to_pickle(BASE + 'data/train_preprocessed.pkl')
test_df.to_pickle(BASE + 'data/test_preprocessed.pkl')
joblib.dump(scaler,        BASE + 'models/scaler.pkl')
joblib.dump(le,            BASE + 'models/label_encoder.pkl')
joblib.dump(class_weights, BASE + 'models/class_weights.pkl')
json.dump({'FEATS': FEATS, 'NUM': NUM, 'CAT': CAT},
          open(BASE + 'data/feat_cols.json', 'w'))

print('Saved:')
for f in ['data/train_preprocessed.pkl',
          'data/test_preprocessed.pkl',
          'data/feat_cols.json',
          'models/scaler.pkl',
          'models/label_encoder.pkl',
          'models/class_weights.pkl']:
    size = os.path.getsize(BASE + f) / 1e6
    print(f'  {f}  —  {size:.1f} MB')

Saving to Drive...
Saved:
  data/train_preprocessed.pkl  —  74.6 MB
  data/test_preprocessed.pkl  —  158.3 MB
  data/feat_cols.json  —  0.0 MB
  models/scaler.pkl  —  0.0 MB
  models/label_encoder.pkl  —  0.0 MB
  models/class_weights.pkl  —  0.0 MB


Fix — run this cell before Step 10

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical columns using combined vocab from train + test
cat_encoders = {}
for col in CAT:
    enc = LabelEncoder()
    # Fit on combined values from both train and test
    combined = pd.concat([train_df[col], test_df[col]]).astype(str)
    enc.fit(combined)
    train_df[col] = enc.transform(train_df[col].astype(str))
    test_df[col]  = enc.transform(test_df[col].astype(str))
    cat_encoders[col] = enc
    print(f'{col}: {list(enc.classes_[:5])} → 0,1,2,3,4...')

print('\nCategorical columns encoded. Ready for CNN and LSTM.')

proto: ['0', '1', '10', '100', '101'] → 0,1,2,3,4...
state: ['0', '1', '10', '2', '3'] → 0,1,2,3,4...
service: ['0', '1', '10', '11', '12'] → 0,1,2,3,4...

Categorical columns encoded. Ready for CNN and LSTM.


Fix both problems — run this complete corrected evaluation cell

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np

# Retrain CNN with corrected setup
print('Rebuilding and retraining CNN...')

# Verify shapes before training
print(f'X_tr shape: {X_tr.shape}')
print(f'X_te shape: {X_te.shape}')
print(f'y_tr shape: {y_tr.shape}')
print(f'y_te shape: {y_te.shape}')
print(f'y_tr unique classes: {np.unique(y_tr)}')
print(f'class_weights: {class_weights}')

Rebuilding and retraining CNN...
X_tr shape: (82332, 42, 1)
X_te shape: (175341, 42, 1)
y_tr shape: (82332,)
y_te shape: (175341,)
y_tr unique classes: [0 1 2 3 4 5 6 7 8 9]
class_weights: {0: np.float64(12.161299852289513), 1: np.float64(14.1221269296741), 2: np.float64(2.013499633162142), 3: np.float64(0.739597556593604), 4: np.float64(1.3581656219069613), 5: np.float64(0.4362884849769488), 6: np.float64(0.22251891891891892), 7: np.float64(2.3550343249427916), 8: np.float64(21.78095238095238), 9: np.float64(187.11818181818182)}


Fix — run this cell to swap them correctly

In [ ]:
# Swap the arrays — train and test were loaded in reverse
X_tr, X_te = X_te, X_tr
y_tr, y_te = y_te, y_tr

# Verify — must show these exact numbers
print(f'X_tr shape: {X_tr.shape}')   # must be (175341, 42, 1)
print(f'X_te shape: {X_te.shape}')   # must be (82332, 42, 1)
print(f'y_tr shape: {y_tr.shape}')   # must be (175341,)
print(f'y_te shape: {y_te.shape}')   # must be (82332,)

X_tr shape: (175341, 42, 1)
X_te shape: (82332, 42, 1)
y_tr shape: (175341,)
y_te shape: (82332,)


Also fix this in your dataframes permanently
Run this too so the serialized text data is also correct for DistilBERT in Session 2:

In [ ]:
# Swap the dataframes too
train_df, test_df = test_df, train_df

print(f'train_df rows: {len(train_df):,}')   # must be 175,341
print(f'test_df rows:  {len(test_df):,}')    # must be 82,332

train_df rows: 175,341
test_df rows:  82,332


Step 10 — Train CNN (E1)

In [ ]:
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras import layers, Model, callbacks
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize

# Prepare arrays
X_tr = train_df[FEATS].values.reshape(-1, len(FEATS), 1).astype('float32')
X_te = test_df[FEATS].values.reshape(-1, len(FEATS), 1).astype('float32')
y_tr = train_df['label_enc'].values
y_te = test_df['label_enc'].values

def build_cnn(n_feats, n_classes=10):
    inp = layers.Input(shape=(n_feats, 1))
    x   = layers.Conv1D(64,  3, activation='relu', padding='same')(inp)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return Model(inp, out)

cnn = build_cnn(len(FEATS))
cnn.compile(optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

es = callbacks.EarlyStopping(monitor='val_accuracy',
                              patience=3,
                              restore_best_weights=True)

print('Training CNN...')
cnn.fit(X_tr, y_tr,
        epochs=10,
        batch_size=1024,
        class_weight=class_weights,
        validation_split=0.1,
        callbacks=[es])

# Evaluate
y_pred_cnn  = np.argmax(cnn.predict(X_te), axis=1)
y_proba_cnn = cnn.predict(X_te)

print('\n=== CNN Results ===')
print(classification_report(y_te, y_pred_cnn, target_names=le.classes_))
auc = roc_auc_score(label_binarize(y_te, classes=range(10)),
                    y_proba_cnn, multi_class='ovr', average='macro')
print(f'AUC-ROC: {auc:.4f}')

# Save
cnn.save(BASE + 'models/cnn_model.h5')
np.save(BASE + 'predictions/cnn_preds.npy', y_pred_cnn)
pd.DataFrame(
    classification_report(y_te, y_pred_cnn,
        target_names=le.classes_, output_dict=True)
).T.to_csv(BASE + 'results/cnn_report.csv')
print('CNN saved.')

Training CNN...
Epoch 1/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 9s 68ms/step - accuracy: 0.2258 - loss: 2.4069 - val_accuracy: 0.0000e+00 - val_loss: 2.3192
Epoch 2/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3667 - loss: 2.0399 - val_accuracy: 0.0000e+00 - val_loss: 2.7537
Epoch 3/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4681 - loss: 1.8691 - val_accuracy: 0.1171 - val_loss: 2.2745
Epoch 4/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5400 - loss: 1.7809 - val_accuracy: 0.1086 - val_loss: 2.6388
Epoch 5/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5734 - loss: 1.6852 - val_accuracy: 0.1727 - val_loss: 2.5741
Epoch 6/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5892 - loss: 1.6350 - val_accuracy: 0.1977 - val_loss: 2.3896
Epoch 7/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6075 - loss: 1.5754 - val_accuracy: 0.2044 - val_loss: 2.1070
Epoch 8/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6111 - loss: 1.5582 - val_acc

AUC-ROC: 0.8638
CNN saved.


The support total is still showing 175,341 which means it evaluated on the training set again, not the test set. The swap didn't fully work. Let's fix this properly by reloading everything from scratch cleanly.

In [ ]:
import pandas as pd, numpy as np, joblib, json, os

# Reload directly from the original CSV files — ignore previous variables
RAW = '/content/unsw_raw/'

train_df = pd.read_csv(RAW + 'UNSW_NB15_training-set.csv', low_memory=False)
test_df  = pd.read_csv(RAW + 'UNSW_NB15_testing-set.csv',  low_memory=False)

print(f'train_df: {len(train_df):,} rows')   # must be 175,341
print(f'test_df:  {len(test_df):,} rows')    # must be 82,332

train_df: 82,332 rows
test_df:  175,341 rows


Run this cell — load with swapped filenames

In [ ]:
import pandas as pd, numpy as np, joblib, json, os
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

RAW = '/content/unsw_raw/'

# Filenames are swapped in this Kaggle upload — load them reversed
train_df = pd.read_csv(RAW + 'UNSW_NB15_testing-set.csv',  low_memory=False)
test_df  = pd.read_csv(RAW + 'UNSW_NB15_training-set.csv', low_memory=False)

print(f'train_df: {len(train_df):,} rows')   # must be 175,341
print(f'test_df:  {len(test_df):,} rows')    # must be 82,332

train_df: 175,341 rows
test_df:  82,332 rows


Then run the full preprocessing again from scratch

In [ ]:
# Feature definitions
DROP  = ['id', 'attack_cat', 'label']
CAT   = ['proto', 'state', 'service']
NUM   = [c for c in train_df.columns if c not in DROP + CAT]
FEATS = CAT + NUM

print(f'Total features: {len(FEATS)}')

# Step 1: Fill missing values
for df in [train_df, test_df]:
    df[NUM] = df[NUM].fillna(df[NUM].median())
    df[CAT] = df[CAT].fillna(df[CAT].mode().iloc[0])

# Step 2: Log1p on skewed features
skewed = [c for c in NUM if train_df[c].skew() > 5.0]
for df in [train_df, test_df]:
    df[skewed] = np.log1p(df[skewed])

# Step 3: MinMax scale — fit on train only
scaler = MinMaxScaler()
train_df[NUM] = scaler.fit_transform(train_df[NUM])
test_df[NUM]  = scaler.transform(test_df[NUM])

# Step 4: Encode labels
le = LabelEncoder()
train_df['label_enc'] = le.fit_transform(train_df['attack_cat'].str.strip())
test_df['label_enc']  = le.transform(test_df['attack_cat'].str.strip())
print('Classes:', le.classes_)

# Step 5: Class weights
cw = compute_class_weight('balanced',
     classes=np.unique(train_df['label_enc']),
     y=train_df['label_enc'])
class_weights = dict(enumerate(cw))
print('\nClass weights:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls}: {class_weights[i]:.2f}')

# Step 6: Encode categoricals for CNN and LSTM
for col in CAT:
    combined = pd.concat([train_df[col], test_df[col]]).astype(str)
    enc = LabelEncoder()
    enc.fit(combined)
    train_df[col] = enc.transform(train_df[col].astype(str))
    test_df[col]  = enc.transform(test_df[col].astype(str))

print('\nCategorical encoding done.')

# Step 7: Serialization for DistilBERT
def serialize(row):
    parts = []
    for col in FEATS:
        val = row[col]
        if isinstance(val, float):
            val = f'{val:.3g}'
        parts.append(f'{col} {val}')
    return ' '.join(parts)

print('Serializing training set...')
train_df['text'] = train_df.apply(serialize, axis=1)
print('Serializing test set...')
test_df['text']  = test_df.apply(serialize, axis=1)

# Step 8: Build arrays for CNN and LSTM
X_tr = train_df[FEATS].values.reshape(-1, len(FEATS), 1).astype('float32')
X_te = test_df[FEATS].values.reshape(-1, len(FEATS), 1).astype('float32')
y_tr = train_df['label_enc'].values
y_te = test_df['label_enc'].values

print(f'\nX_tr: {X_tr.shape}')   # must be (175341, 42, 1)
print(f'X_te: {X_te.shape}')    # must be (82332, 42, 1)

Total features: 42
Classes: ['Analysis' 'Backdoor' 'DoS' 'Exploits' 'Fuzzers' 'Generic' 'Normal'
 'Reconnaissance' 'Shellcode' 'Worms']

Class weights:
  Analysis: 8.77
  Backdoor: 10.04
  DoS: 1.43
  Exploits: 0.53
  Fuzzers: 0.96
  Generic: 0.44
  Normal: 0.31
  Reconnaissance: 1.67
  Shellcode: 15.48
  Worms: 134.88

Categorical encoding done.
Serializing training set...
Serializing test set...

X_tr: (175341, 42, 1)
X_te: (82332, 42, 1)


Then save everything to Drive again with correct data

In [ ]:
train_df.to_pickle(BASE + 'data/train_preprocessed.pkl')
test_df.to_pickle(BASE + 'data/test_preprocessed.pkl')
joblib.dump(scaler,        BASE + 'models/scaler.pkl')
joblib.dump(le,            BASE + 'models/label_encoder.pkl')
joblib.dump(class_weights, BASE + 'models/class_weights.pkl')
json.dump({'FEATS': FEATS, 'NUM': NUM, 'CAT': CAT},
          open(BASE + 'data/feat_cols.json', 'w'))

print('All files saved to Drive correctly.')

All files saved to Drive correctly.


Then retrain CNN

In [ ]:
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras import layers, Model, callbacks
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize

def build_cnn(n_feats, n_classes=10):
    inp = layers.Input(shape=(n_feats, 1))
    x   = layers.Conv1D(64,  3, activation='relu', padding='same')(inp)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Conv1D(256, 3, activation='relu', padding='same')(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(64,  activation='relu')(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return Model(inp, out)

cnn = build_cnn(len(FEATS))
cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

es = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print('Training CNN on correct data...')
cnn.fit(
    X_tr, y_tr,
    epochs=10,
    batch_size=1024,
    class_weight=class_weights,
    validation_split=0.1,
    callbacks=[es],
    verbose=1
)

# Evaluate on test set
y_proba_cnn = cnn.predict(X_te)
y_pred_cnn  = np.argmax(y_proba_cnn, axis=1)

print(f'\nTest set size: {len(y_te):,}')   # must show 82,332

print('\n=== CNN Results (TEST SET) ===')
print(classification_report(y_te, y_pred_cnn, target_names=le.classes_))

auc = roc_auc_score(
    label_binarize(y_te, classes=range(10)),
    y_proba_cnn,
    multi_class='ovr',
    average='macro'
)
print(f'AUC-ROC: {auc:.4f}')

# Save with new keras format
cnn.save(BASE + 'models/cnn_model.keras')
np.save(BASE + 'predictions/cnn_preds.npy', y_pred_cnn)
pd.DataFrame(
    classification_report(y_te, y_pred_cnn,
        target_names=le.classes_, output_dict=True)
).T.to_csv(BASE + 'results/cnn_report.csv')
print('CNN saved.')

Training CNN on correct data...
Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 26s 113ms/step - accuracy: 0.1333 - loss: 2.3739 - val_accuracy: 0.0027 - val_loss: 2.4368
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1978 - loss: 2.3498 - val_accuracy: 0.0563 - val_loss: 2.5277
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2143 - loss: 2.2677 - val_accuracy: 0.0258 - val_loss: 2.2827
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2610 - loss: 2.1290 - val_accuracy: 0.6524 - val_loss: 1.9180
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4005 - loss: 1.9996 - val_accuracy: 0.7760 - val_loss: 1.5056
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4778 - loss: 1.8620 - val_accuracy: 0.7948 - val_loss: 1.1790
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5104 - loss: 1.7734 - val_accuracy: 0.8175 - val_loss: 0.8160
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.529

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


CNN saved.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Now run LSTM

In [ ]:
from tensorflow.keras import layers, Model, callbacks
import tensorflow as tf
tf.random.set_seed(42)

def build_lstm(n_feats, n_classes=10):
    inp = layers.Input(shape=(n_feats, 1))
    x   = layers.LSTM(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.LSTM(64)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return Model(inp, out)

lstm = build_lstm(len(FEATS))
lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

es = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print('Training LSTM (~12 min on A100)...')
lstm.fit(
    X_tr, y_tr,
    epochs=10,
    batch_size=512,
    class_weight=class_weights,
    validation_split=0.1,
    callbacks=[es],
    verbose=1
)

# Evaluate on test set
y_proba_lstm = lstm.predict(X_te)
y_pred_lstm  = np.argmax(y_proba_lstm, axis=1)

print(f'\nTest set size: {len(y_te):,}')   # must show 82,332

print('\n=== LSTM Results (TEST SET) ===')
print(classification_report(y_te, y_pred_lstm, target_names=le.classes_))

auc_lstm = roc_auc_score(
    label_binarize(y_te, classes=range(10)),
    y_proba_lstm,
    multi_class='ovr',
    average='macro'
)
print(f'AUC-ROC: {auc_lstm:.4f}')

# Save
lstm.save(BASE + 'models/lstm_model.keras')
np.save(BASE + 'predictions/lstm_preds.npy', y_pred_lstm)
pd.DataFrame(
    classification_report(y_te, y_pred_lstm,
        target_names=le.classes_, output_dict=True)
).T.to_csv(BASE + 'results/lstm_report.csv')
print('LSTM saved. Session 1 complete.')

Training LSTM (~12 min on A100)...
Epoch 1/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.1655 - loss: 2.2822 - val_accuracy: 0.7484 - val_loss: 1.3986
Epoch 2/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.3727 - loss: 2.0247 - val_accuracy: 0.7837 - val_loss: 0.7299
Epoch 3/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.3943 - loss: 1.8858 - val_accuracy: 0.7964 - val_loss: 0.6277
Epoch 4/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.4145 - loss: 1.7978 - val_accuracy: 0.7924 - val_loss: 0.6022
Epoch 5/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.4264 - loss: 1.7673 - val_accuracy: 0.7950 - val_loss: 0.5908
Epoch 6/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.4344 - loss: 1.7346 - val_accuracy: 0.7974 - val_loss: 0.5874
Epoch 7/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.4409 - loss: 1.7134 - val_accuracy: 0.8018 - val_loss: 0.5640
Epoch 8/10
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accurac

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


LSTM saved. Session 1 complete.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Before moving to Session 2 — verify Drive is complete

In [ ]:
import os

required = [
    'data/train_preprocessed.pkl',
    'data/test_preprocessed.pkl',
    'data/feat_cols.json',
    'models/scaler.pkl',
    'models/label_encoder.pkl',
    'models/class_weights.pkl',
    'models/cnn_model.keras',
    'models/lstm_model.keras',
    'predictions/cnn_preds.npy',
    'predictions/lstm_preds.npy',
    'results/cnn_report.csv',
    'results/lstm_report.csv',
]

print('Checking Drive files...')
all_good = True
for f in required:
    path = BASE + f
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1e6 if exists else 0
    status = f'OK  {size:.1f} MB' if exists else 'MISSING'
    print(f'  {f:45s} {status}')
    if not exists:
        all_good = False

print()
print('Session 1 complete.' if all_good else 'Some files missing — do not proceed to Session 2.')

Checking Drive files...
  data/train_preprocessed.pkl                   OK  159.4 MB
  data/test_preprocessed.pkl                    OK  75.1 MB
  data/feat_cols.json                           OK  0.0 MB
  models/scaler.pkl                             OK  0.0 MB
  models/label_encoder.pkl                      OK  0.0 MB
  models/class_weights.pkl                      OK  0.0 MB
  models/cnn_model.keras                        OK  2.0 MB
  models/lstm_model.keras                       OK  1.5 MB
  predictions/cnn_preds.npy                     OK  0.7 MB
  predictions/lstm_preds.npy                    OK  0.7 MB
  results/cnn_report.csv                        OK  0.0 MB
  results/lstm_report.csv                       OK  0.0 MB

Session 1 complete.
